# COMP64702 Coursework — RAG Pipeline

This notebook provides a high-level walkthrough of the RAG pipeline implemented for this coursework.

It serves as a single entry point to demonstrate that the entire pipeline can be executed end-to-end.

To improve efficiency, intermediate results are already provided (e.g., processed corpus, chunks, index, and generated outputs), so most steps can be skipped. However, each stage can also be re-run if needed.

**Note:**  
This notebook is intended as a concise demonstration of the pipeline rather than a full code explanation.  
For detailed implementation and design decisions of each component, please refer to the `README.md` and the source code in the `src/` directory.

---

## Pipeline Overview

1. Setup  
2. Project structure and paths  
3. Data preparation  
4. Corpus cleaning  
5. Chunking  
6. Benchmark creation  
7. Index building  
8. Retrieval / Generation  
9. Evaluation  

## 0. Environment and Running Notes

To run it successfully:
- use Python 3.10+ (recommended)
- install the required packages before running the pipeline
- note that the first execution may download models from HuggingFace, which can take additional time depending on network conditions

If the environment is not ready, run the optional installation cell below first.

In [20]:
# Optional: install required packages
# Run this cell only if the environment is not yet set up.

!pip install -r requirements.txt


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
# Optional: preload models (may take a few minutes)

from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer

print("Downloading embedding model...")
SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Downloading generation model...")
AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
AutoModel.from_pretrained("Qwen/Qwen2.5-0.5B")

print("Models downloaded successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 919.65it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 879.89it/s, Materializing param=norm.weight]                              


Models downloaded successfully.


## 1. Setup

This notebook is the main entry point for the submitted RAG pipeline.
It orchestrates the major stages of the system, including data preparation,
chunking, benchmark creation, indexing, generation, and evaluation.

In [22]:
import os
import sys
from pathlib import Path

# Set project root
PROJECT_ROOT = Path.cwd()

# If notebook is placed in another folder, adjust here if needed
DATA_DIR = PROJECT_ROOT / "data"
SRC_DIR = PROJECT_ROOT / "src"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

print("Project root:", PROJECT_ROOT)
print("Data dir exists:", DATA_DIR.exists())
print("Src dir exists:", SRC_DIR.exists())
print("Scripts dir exists:", SCRIPTS_DIR.exists())

Project root: d:\File\T2M\RAG\COMP64702\rag_project
Data dir exists: True
Src dir exists: True
Scripts dir exists: True


In [23]:
# Make sure Python can import from src
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(sys.path[-3:])

['', 'd:\\File\\T2M\\RAG\\COMP64702\\rag_project\\.venv\\Lib\\site-packages', 'd:\\File\\T2M\\RAG\\COMP64702\\rag_project']


## 2. Project structure and paths

The project is organised into the following modules:

- `src/ingestion/`: data fetching, cleaning, chunking, and indexing
- `scripts/`: benchmark creation
- `src/retrieval/`: retrieval utilities
- `src/generation/`: prompting and answer generation
- `src/evaluation/`: evaluation scripts

Intermediate and final outputs are stored under the `data/` directory.

## 3. Data preparation

The cleaned corpus is already provided in:

`data/processed/corpus.jsonl`

Therefore, this step can be skipped to save time.

In [24]:
# Check key intermediate files
corpus_file = DATA_DIR / "processed" / "corpus.jsonl"
print("Corpus exists:", corpus_file.exists())

Corpus exists: True


If re-running the data preparation stage is desired, please execute the following steps:

1. Run `fetch.py` to collect raw data and store it in `data/raw/`
2. Run `clean.py` to process the raw data and generate the cleaned corpus in `data/processed/corpus.jsonl`

In [25]:
# Optional: re-run data collection
!python -m src.ingestion.fetch

[1/12] Fetching: https://en.wikipedia.org/wiki/Mediterranean_cuisine
      Saved -> data\raw\wiki__mediterranean_cuisine.json (HTTP 200)
[2/12] Fetching: https://en.wikipedia.org/wiki/Mediterranean_diet
      Saved -> data\raw\wiki__mediterranean_diet.json (HTTP 200)
[3/12] Fetching: https://en.wikipedia.org/wiki/Portuguese_cuisine
      Saved -> data\raw\wiki__portuguese_cuisine.json (HTTP 200)
[4/12] Fetching: https://en.wikipedia.org/wiki/Spanish_cuisine
      Saved -> data\raw\wiki__spanish_cuisine.json (HTTP 200)
[5/12] Fetching: https://en.wikipedia.org/wiki/Provence#Cuisine
      Saved -> data\raw\wiki__provence.json (HTTP 200)
[6/12] Fetching: https://en.wikipedia.org/wiki/Occitan_cuisine
      Saved -> data\raw\wiki__occitan_cuisine.json (HTTP 200)
[7/12] Fetching: https://en.wikipedia.org/wiki/Italian_cuisine
      Saved -> data\raw\wiki__italian_cuisine.json (HTTP 200)
[8/12] Fetching: https://en.wikipedia.org/wiki/Greek_cuisine
      Saved -> data\raw\wiki__greek_cuisine.js

In [26]:
# Optional: re-run data cleaning
!python -m src.ingestion.clean

USING UPDATED CLEAN.PY
Corpus built → D:\File\T2M\RAG\COMP64702\rag_project\data\processed\corpus.jsonl
Processed 58 raw file(s)
Total blocks written: 854
Blocks per source: {'blog80cuisines': 44, 'wikipedia': 728, 'wikibooks': 82}
Exact duplicates removed: 47
Average block length: 527.29 chars
Median block length: 410.50 chars
Max block length: 5639 chars
Blocks > 1200 chars: 57


## 4. Chunking

The chunked corpus is already provided in:

`data/processed/chunks_v3.jsonl`

Therefore, this step can be skipped to save time.

In [27]:
chunks_file = DATA_DIR / "processed" / "chunks_v3.jsonl"
print("Chunks exist:", chunks_file.exists())

Chunks exist: True


If re-running the chunking stage is desired, execute the chunking script to regenerate the chunked corpus from the cleaned corpus.

In [28]:
# Optional: re-run chunking
!python -m src.ingestion.chunking

 Successfully loaded 854 documents.

Starting chunking...
Removing duplicate chunks...
Removed 15 duplicates
Done! 854 documents were converted into 1263 chunks.
Final data saved to: data\processed\chunks_v3.jsonl


d:\File\T2M\RAG\COMP64702\rag_project\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


## 5. Benchmark creation

The benchmark files are already provided in:

- `data/benchmark/benchmark.json`
- `data/benchmark/input_payload.json`

Therefore, this step can be skipped to save time.

In [29]:
benchmark_file = DATA_DIR / "benchmark" / "benchmark.json"
input_payload_file = DATA_DIR / "benchmark" / "input_payload.json"
print("Benchmark exists:", benchmark_file.exists())
print("input_payload exists:", input_payload_file.exists())

Benchmark exists: True
input_payload exists: True


If re-running the benchmark creation stage is desired, execute `create_benchmark.py` to regenerate the benchmark and the corresponding input payload.

In [30]:
# Optional: re-run benchmark creation
!python scripts/create_benchmark.py

Benchmark saved to data\benchmark\benchmark.json
Input payload saved to data\benchmark\input_payload.json
Total benchmark items: 50


## 6. Index building

This stage builds the vector index used in the RAG system.

The retrieval index is already provided in:

`data/retrieval/index_multiqa.json`

Therefore, this step can be skipped to save time.

In [31]:
index_file = DATA_DIR / "retrieval" / "index_multiqa.json"
print("Index exists:", index_file.exists())

Index exists: True


If re-running the index building stage is desired, execute `build_index.py` to regenerate the vector index from the chunked corpus.

In [32]:
# Optional: re-run index building
!python -m src.ingestion.build_index

Loading chunks from: D:\File\T2M\RAG\COMP64702\rag_project\data\processed\chunks_v3.jsonl
Loaded 1263 chunks.
Saved index to: D:\File\T2M\RAG\COMP64702\rag_project\data\retrieval\index_multiqa.json
Number of indexed chunks: 1263
Embedding dimension: 768



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4869.05it/s, Materializing param=pooler.dense.weight]
MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 7. Retrieval / Generation

This stage performs retrieval and answer generation in the RAG pipeline.

### 7.1 Retrieval

The retriever component can be tested independently using:

`src/retrieval/retriever.py`

This allows inspection of the top retrieved chunks for a given query.
 



In [33]:
# Optional: test retriever with a sample query
!python -m src.retrieval.retriever

Query: What are common ingredients in Mediterranean cuisine?
Top 5 reranked results:

Result 1
Chunk ID: chunk_642
Dense score: 0.7052
Lexical score: 0.7143
Final score: 0.9552
Text: The ingredients of Mediterranean cuisine are to an extent different from those of the cuisine of Northern Europe, with olive oil instead of butter, and wine instead of beer. The list of available ingredients has changed over the centuries. One major change was the introduction of many foods by the A
Metadata: {'doc_id': 'wiki__mediterranean_cuisine__p28', 'source': 'wikipedia', 'url': 'https://en.wikipedia.org/wiki/Mediterranean_cuisine', 'title': 'Mediterranean cuisine', 'section_path': ['History', 'Origins'], 'page_type': 'article', 'char_len': 503, 'block_type': 'paragraph', 'source_doc_id': 'wiki__mediterranean_cuisine__p28', 'chunk_index': 0, 'chunk_method': 'keep_whole'}

Result 2
Chunk ID: chunk_648
Dense score: 0.7350
Lexical score: 0.5714
Final score: 0.9350
Text: With common ingredients including


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5049.13it/s, Materializing param=pooler.dense.weight]
MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 7.2 Generation

A generated output file is already provided in:

`data/output/output_strict_fallback_rescue.json`

This file contains the model predictions and can be used directly for evaluation. Therefore, this step can be skipped to save time.

In [34]:
output_file = DATA_DIR / "output" / "output_strict_fallback_rescue.json"
print("The generated output exists:", output_file.exists())

The generated output exists: True


If re-running the generation stage is desired, execute the following cell to produce a fresh output file, for example:

`data/output/output_strict_example.json`

In [35]:
# Optional: re-run generation.
# Note: this step may take approximately 20–30 minutes depending on hardware.
# A pre-generated output file is already provided for evaluation.

!python -m src.generation.demo_prompting data/benchmark/input_payload.json data/output/output_strict_example.json strict

Processing 1/50: The name 'bouillabaisse' is derived from which two words?
Processing 2/50: Which two ingredients are traditionally not used in authenti
Processing 3/50: What is chermoula?
Processing 4/50: What is passata in Italian cuisine?
Processing 5/50: What are the three core components of the Mediterranean diet
Processing 6/50: In Trapani (Sicily), which North African-influenced dish is 
Processing 7/50: What dish is traditionally eaten on Thursdays in the Lazio r
Processing 8/50: What health benefit is commonly associated with drinking kar
Processing 9/50: How are French and Italian cuisines contrasted?
Processing 10/50: Which plant is known as the bread tree in Mediterranean moun
Processing 11/50: What type of flour is commonly used to fry fish in the Black
Processing 12/50: What are the typical fillings of su böreği in Turkish cuisin
Processing 13/50: Why did the Romans forbid growing vines in Provence in 120 B
Processing 14/50: Why can olive oil replace saturated fats?
Proce


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 4896.51it/s, Materializing param=model.norm.weight]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4726.69it/s, Materializing param=pooler.dense.weight]


## Final pipeline run

Run the final pipeline in strict mode using `data/benchmark/input_payload.json` and save the result to `data/output/output_payload.json`.

In [1]:
from src.generation.demo_prompting import load_model, load_retriever, run_payload

In [2]:
tokenizer, model = load_model()
retriever = load_retriever()
print("Model and retriever loaded.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model and retriever loaded.


In [3]:
run_payload(
    input_path="data/benchmark/input_payload.json",
    output_path="data/output/output_payload.json",
    prompt_mode="strict",
    tokenizer=tokenizer,
    model=model,
    retriever=retriever
)

Processing 1/50: The name 'bouillabaisse' is derived from which two words?
Processing 2/50: Which two ingredients are traditionally not used in authenti
Processing 3/50: What is chermoula?
Processing 4/50: What is passata in Italian cuisine?
Processing 5/50: What are the three core components of the Mediterranean diet
Processing 6/50: In Trapani (Sicily), which North African-influenced dish is 
Processing 7/50: What dish is traditionally eaten on Thursdays in the Lazio r
Processing 8/50: What health benefit is commonly associated with drinking kar
Processing 9/50: How are French and Italian cuisines contrasted?
Processing 10/50: Which plant is known as the bread tree in Mediterranean moun
Processing 11/50: What type of flour is commonly used to fry fish in the Black
Processing 12/50: What are the typical fillings of su böreği in Turkish cuisin
Processing 13/50: Why did the Romans forbid growing vines in Provence in 120 B
Processing 14/50: Why can olive oil replace saturated fats?
Proce

In [1]:
import subprocess
import sys
import os

cmd = [
    sys.executable,
    "-u",
    "-m",
    "src.generation.demo_prompting",
    "data/benchmark/input_payload.json",
    "data/output/output_payload.json",
    "strict",
]

env = os.environ.copy()
env["PYTHONUTF8"] = "1"

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    env=env,
    bufsize=1
)

for line in process.stdout:
    print(line, end="")

process.wait()
print("return code:", process.returncode)


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2407.44it/s, Materializing param=model.norm.weight]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2599.57it/s, Materializing param=pooler.dense.weight]
Processing 1/50: The name 'bouillabaisse' is derived from which two words?
Processing 2/50: Which two ingredients are traditionally not used in authenti
Processing 3/50: What is chermoula?
Processing 4/50: What is passata in Italian cuisine?
Processing 5/50: What are the three core components of the Mediterranean diet
Processing 6/50: In Trapani (Sicily), which North African-influenced dish is 
Processing 7/50: What dish is traditionally eaten on Thursdays in the Lazio r
Processing 8/50: What health benefit is commonly associated with drinking kar
Processing 9/50: How are French and Italian cuisines contrasted?
Processing 10/50: Which plant is known as the bread tree in Mediterranean moun
Processing 11/50: What type of flour is commonly used to fry fish in the Black
Pro

In [37]:
import json
from pathlib import Path

output_path = Path("data/output/output_payload.json")

print("Output exists:", output_path.exists())

if output_path.exists():
    with open(output_path, "r", encoding="utf-8") as f:
        output_payload = json.load(f)

    print("Number of results:", len(output_payload.get("results", [])))
    if output_payload.get("results"):
        print(json.dumps(output_payload["results"][0], ensure_ascii=False, indent=2)[:1000])

Output exists: True
Number of results: 50
{
  "query_id": "q_000",
  "query": "The name 'bouillabaisse' is derived from which two words?",
  "response": "The name 'bouillabaisse' is derived from the words 'bolhir' (boil) and 'abaissar' (simmer).",
  "retrieved_context": [
    {
      "doc_id": "blog80__aroundtheworldin80cuisinesblog_wordpress_com_2017_04_14_1_france_and_monaco__p1",
      "text": "The name bouillabaisse stems from a combination of words “bolhir” (to boil) and “abaissar” (to simmer), referring to the process of first boiling the broth, then adding each type of fish one by one, returning the pot to a simmer between each addition"
    },
    {
      "doc_id": "wiki__mediterranean_cuisine__p48",
      "text": "Bouillabaisse is a substantial dish from the French port of Marseille, capital of Provence. It is a stew for at least eight people, because it should contain many types of fish such as crayfish, gurnard, weever, John Dory, monkfish, conger eel, whiting, sea bass, and

## 8. Evaluation

This stage evaluates both the retrieval and generation performance of the RAG system.

All evaluation outputs are saved under the `data/eval/` directory, including:

- `retrieval_eval_results.json` for retrieval evaluation
- `evaluation_summary.json` for overall generation performance
- `evaluation_by_type.json` for per-question-type analysis
- `evaluation_error_cases.json` for detailed error inspection

### 8.1 Retrieval Evaluation

The retrieval component is evaluated using:

`src/evaluation/eval_pipeline.py`

This evaluation measures how well the retriever identifies relevant chunks for each query.

In [1]:
# Run retrieval evaluation
!python -m src.evaluation.eval_pipeline


[Overall]
Count: 50
Hit@1: 0.740
Hit@3: 0.860
Hit@5: 0.860

[Answerable Only]
Count: 45
Hit@1: 0.822
Hit@3: 0.956
Hit@5: 0.956

[Difficulty = easy]
Count: 19
Hit@1: 0.737
Hit@3: 1.000
Hit@5: 1.000

[Difficulty = medium]
Count: 20
Hit@1: 0.850
Hit@3: 0.900
Hit@5: 0.900

[Difficulty = hard]
Count: 6
Hit@1: 1.000
Hit@3: 1.000
Hit@5: 1.000

[Difficulty = negative]
Count: 5
Hit@1: 0.000
Hit@3: 0.000
Hit@5: 0.000

Detailed results saved to: F:\Text\COMP64702_clean\rag_project\data\eval\retrieval_eval_results.json



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2773.80it/s, Materializing param=pooler.dense.weight]
MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



### 8.2 Generation Evaluation

The generation component is evaluated using:

`src/evaluation/eval_pipeline2.py`

The following evaluation uses the pre-generated output file:

`data/output/output_payload.json`


In [2]:
!python -m src.evaluation.eval_pipeline2

=== Generation Evaluation ===
Answerable Questions: 45
Unanswerable Questions: 5
Exact Match Accuracy: 0.04
Keyword Match Accuracy: 0.16
Average Token F1: 0.25
Unanswerable Accuracy: 0.00
IDK Precision: 0.00
IDK Recall: 0.00
IDK F1: 0.00

=== By Question Type ===

[factoid]
  count: 15
  exact_match_accuracy: 0.13
  keyword_match_accuracy: 0.20
  average_token_f1: 0.35
  idk_rate: 0.00

[which]
  count: 2
  exact_match_accuracy: 0.00
  keyword_match_accuracy: 0.00
  average_token_f1: 0.00
  idk_rate: 0.00

[definition]
  count: 7
  exact_match_accuracy: 0.00
  keyword_match_accuracy: 0.29
  average_token_f1: 0.29
  idk_rate: 0.00

[how]
  count: 8
  exact_match_accuracy: 0.00
  keyword_match_accuracy: 0.12
  average_token_f1: 0.20
  idk_rate: 0.00

[why]
  count: 10
  exact_match_accuracy: 0.00
  keyword_match_accuracy: 0.00
  average_token_f1: 0.20
  idk_rate: 0.00

[comparison]
  count: 3
  exact_match_accuracy: 0.00
  keyword_match_accuracy: 0.33
  average_token_f1: 0.18
  idk_rate:

The evaluation script can be pointed to either:
- `data/output/output_payload.json`, or
- `data/output/output_strict_example.json`

by changing the prediction file path inside the evaluation script.